In [24]:
from pathlib import Path
import pandas as pd
import numpy as np
from ChildProject.projects import ChildProject
from ChildProject.annotations import AnnotationManager
import ast

DATA_PATH = Path('/home/engaclew/neurogen')
columns = ['recording_filename', '5s_CTC', 'voc_chi', 'voc_dur_chi']

# Read measures
aclew_measures = pd.read_csv(DATA_PATH / 'aclew_measures_chunks.csv').fillna(0)
lena_measures = pd.read_csv(DATA_PATH / 'lena_measures_chunks.csv').fillna(0)
human_measures = pd.read_csv(DATA_PATH / 'human_measures_chunks.csv').fillna(0)

# Read metadata
children = pd.read_csv(DATA_PATH / 'data/L3_HIPAA_LENA_cleaned/metadata/children.csv')
recordings = pd.read_csv(DATA_PATH / 'data/L3_HIPAA_LENA_cleaned/metadata/recordings.csv')
recordings_data = recordings.merge(children, on='child_id')[['group_id', 'recording_filename']]
aclew_measures = aclew_measures.merge(recordings_data, how='left', on='recording_filename')
lena_measures = lena_measures.merge(recordings_data, how='left', on='recording_filename')
human_measures = human_measures.merge(recordings_data, how='left', on='recording_filename')

def compute_CVC(data):
    if 'can_voc_CHI' in data.columns and 'non_can_voc_CHI' in data.columns:
        data['CVC'] = data['can_voc_CHI'] + data['non_can_voc_CHI']
    return data

aclew_measures = compute_CVC(aclew_measures)
human_measures = compute_CVC(human_measures)
# LENA already has CVC

# Add pitch
# Merge pitch from speech-like and non-speech-like vocalizations
def clean_pitch_concatenation(df):
    def safe_eval(x):
        if pd.isna(x):
            return []
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return []
    df['speechlike_pitch'] = df['speechlike_pitch'].apply(safe_eval)
    df['nonspeechlike_pitch'] = df['nonspeechlike_pitch'].apply(safe_eval)
    df['pitch'] = df.apply( lambda row: row['speechlike_pitch'] + row['nonspeechlike_pitch'],  axis=1)
    return df
human_measures = clean_pitch_concatenation(human_measures)
human_measures['speechlike_voc_dur_CHI'] = human_measures['can_voc_dur_CHI'] + human_measures['non_can_voc_dur_CHI']

def safe_median(x):
        return np.nan if len(x) == 0 else np.median(x)
human_measures['median_pitch'] = human_measures['pitch'].map(safe_median)
human_measures = human_measures.rename(columns={'5s_CTC': 'CTC','wc_adu': 'AWC'})
aclew_measures = aclew_measures.rename(columns={'5s_CTC': 'CTC','wc_adu': 'AWC'})
lena_measures = lena_measures.rename(columns={'5s_CTC': 'CTC','wc_adu': 'AWC'})
# Compute mean and mean percentage error
def compute_me(x, y):
    """ Compute mean error """
    me_list = (x-y).values
    return me_list

def compute_mape(x, y):
    """ Compute mean error """
    errors = np.zeros_like(x, dtype=float)
    # Case 1: y != 0 - standard MPE calculation
    mask_nonzero = (y != 0)
    errors[mask_nonzero] = (x[mask_nonzero] - y[mask_nonzero]) / y[mask_nonzero]
    # Case 2: y = 0 and x != 0 -> error = 1
    mask_zero_error = (y == 0) & (x != 0)
    errors[mask_zero_error] = 1
    # Case 3: y = 0 and x = 0 -> error = 0 is already handled by initialization
    return 100*np.abs(errors)

cols = ['CTC', 'AWC', 'CVC']
lena_scores, aclew_scores = pd.DataFrame(), pd.DataFrame()
for col in cols:
    lena_scores[f'{col}_mape'] = compute_mape(lena_measures[col], human_measures[col])
    aclew_scores[f'{col}_mape'] = compute_mape(aclew_measures[col], human_measures[col])

duration_cols = ['voc_dur_fem', 'voc_dur_mal', 'voc_dur_och', 'voc_dur_chi', 'speechlike_voc_dur_CHI', 'can_voc_dur_CHI', 'non_can_voc_dur_CHI', 'cry_voc_dur_CHI']

for duration_col in duration_cols:
    human_measures[duration_col] /= 1000.0

human_measures['recording_filename'] = human_measures['recording_filename'].map(lambda x: x.replace('.wav', ''))
human_measures['clip_id'] = human_measures['recording_filename'] + '_' + human_measures['segment_onset'].astype(str) + '_' + human_measures['segment_offset'].astype(str)

copy_cols = ['clip_id', 'group_id', 'child_id', 'voc_dur_fem', 'voc_dur_mal', 'voc_dur_och', 'voc_dur_chi', 
             'AWC', 'CTC', 'overlap_dur', 'CVC', 'speechlike_voc_dur_CHI', 'can_voc_dur_CHI', 'non_can_voc_dur_CHI', 'cry_voc_dur_CHI']
for col in copy_cols:
    lena_scores[col] = human_measures[col]
    aclew_scores[col] = human_measures[col]


# Finally add identification error rate components and percentage correct
results_folder = DATA_PATH / 'results' / 'pyannote_metrics'
ider_vtc = pd.read_csv(results_folder / 'vtc_eaf_an1' / 'ider_2mn_clips.csv')
ider_lena = pd.read_csv(results_folder / 'its_eaf_an1' / 'ider_2mn_clips.csv')

def compute_ider(ider_df):
    ider_df['missed detection'] = 100*ider_df['missed detection'] / ider_df['total']
    ider_df['false alarm'] = 100*ider_df['false alarm'] / ider_df['total']
    ider_df['confusion'] = 100*ider_df['confusion'] / ider_df['total']
    ider_df['correct'] = 100*ider_df['correct'] / ider_df['total']
    ider_df['ider'] = 100*ider_df['ider']
    return ider_df

ider_vtc = compute_ider(ider_vtc)
ider_lena = compute_ider(ider_lena)
ider_vtc['recording_id'] = ider_vtc['recording_id'].map(lambda x: x.replace('.wav', ''))
ider_lena['recording_id'] = ider_lena['recording_id'].map(lambda x: x.replace('.wav', ''))
ider_vtc['clip_id'] = ider_vtc['recording_id'] + '_' + ider_vtc['onset'].astype(str) + '_' + ider_vtc['offset'].astype(str)
ider_lena['clip_id'] = ider_lena['recording_id'] + '_' + ider_lena['onset'].astype(str) + '_' + ider_lena['offset'].astype(str)

                                                        
cols = ['clip_id', 'confusion', 'correct', 'missed detection', 'false alarm']
ider_vtc = ider_vtc[cols]
ider_lena = ider_lena[cols]
rename_map = {'missed detection': 'miss', 'false alarm': 'false_alarm'}
ider_vtc = ider_vtc.rename(columns=rename_map)
ider_lena = ider_lena.rename(columns=rename_map)


aclew_scores = aclew_scores.merge(ider_vtc, on='clip_id')
lena_scores = lena_scores.merge(ider_lena, on='clip_id')


In [ ]:
import statsmodels.formula.api as smf
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from scipy import stats

def analyze_mixed_model_anova(data, dvs, predictors):
    """
    Run mixed models with Type II tests for each predictor
    """
    warnings.filterwarnings('ignore', category=ConvergenceWarning)
    
    for dv in dvs:
        print(f"\n{'='*80}")
        print(f"=== {dv} ===")
        print(f"{'='*80}")
        
        try:
            # Prepare data
            model_data = data[[dv] + predictors + ['child_id']].dropna()
            
            if len(model_data) == 0:
                print(f"No complete data for {dv}")
                continue
            
            # Fit full model with all predictors
            predictor_string = " + ".join(predictors)
            formula = f"{dv} ~ {predictor_string}"
            
            full_model = smf.mixedlm(
                formula,
                data=model_data,
                groups="child_id"
            ).fit()
            
            # Calculate marginal R²
            var_fixed = np.var(full_model.fittedvalues)
            var_random = float(full_model.cov_re.iloc[0, 0])
            var_residual = full_model.scale
            total_var = var_fixed + var_random + var_residual
            marginal_r2 = var_fixed / total_var
            
            print(f"\nFull Model R² (marginal) = {marginal_r2*100:.1f}%, n = {len(model_data)}")
            
            # For each predictor, compare models with and without it (Type II test)
            results = []
            
            for predictor in predictors:
                # Fit reduced model without this predictor
                predictors_without = [p for p in predictors if p != predictor]
                predictor_string_reduced = " + ".join(predictors_without)
                formula_reduced = f"{dv} ~ {predictor_string_reduced}"
                
                reduced_model = smf.mixedlm(
                    formula_reduced,
                    data=model_data,
                    groups="child_id"
                ).fit(reml=False)
                
                # Refit full model with ML for comparison
                full_model_ml = smf.mixedlm(
                    formula,
                    data=model_data,
                    groups="child_id"
                ).fit(reml=False)
                
                # Likelihood ratio test
                lr_stat = 2 * (full_model_ml.llf - reduced_model.llf)
                df_diff = 1  # One predictor
                p_value = stats.chi2.sf(lr_stat, df_diff)
                
                # Get coefficient from full model (REML version)
                coef = full_model.params[predictor]
                
                # Calculate semi-partial R² (unique contribution)
                var_fixed_reduced = np.var(reduced_model.fittedvalues)
                var_random_reduced = float(reduced_model.cov_re.iloc[0, 0])
                var_residual_reduced = reduced_model.scale
                total_var_reduced = var_fixed_reduced + var_random_reduced + var_residual_reduced
                marginal_r2_reduced = var_fixed_reduced / total_var_reduced
                
                semipartial_r2 = marginal_r2 - marginal_r2_reduced
                
                # Only keep if significant
                if p_value < 0.05:
                    # Format significance
                    if p_value < 0.001:
                        p_str = "p<.001***"
                    elif p_value < 0.01:
                        p_str = f"p={p_value:.3f}**"
                    else:
                        p_str = f"p={p_value:.3f}*"
                    
                    results.append({
                        'Predictor': predictor,
                        'β': coef,
                        'sr²': semipartial_r2 * 100,
                        'p-value': p_str,
                        'p_raw': p_value
                    })
            
            # Display results
            if results:
                results_df = pd.DataFrame(results).sort_values('sr²', ascending=False)
                print("\nSignificant Predictors:")
                print(f"{'Predictor':<25} {'β':>10} {'sr²':>10} {'p-value':>15}")
                print("-" * 65)
                for _, row in results_df.iterrows():
                    print(f"{row['Predictor']:<25} {row['β']:>10.3f} {row['sr²']:>9.1f}% {row['p-value']:>15}")
            else:
                print("\nNo significant predictors")
                
        except Exception as e:
            print(f"Error analyzing {dv}: {str(e)}")
            import traceback
            traceback.print_exc()

# Define predictors and DVs
predictors = ['voc_dur_fem', 'voc_dur_mal', 'voc_dur_och', 'voc_dur_chi']

dvs = ['miss', 'false_alarm', 'confusion', 'correct', 
       'CTC_mape', 'AWC_mape', 'CVC_mape']

# Run analysis
print("\n" + "="*80)
print("LENA®")
print("="*80)
analyze_mixed_model_anova(lena_scores, dvs, predictors)

print("\n" + "="*80)
print("ACLEW")
print("="*80)
analyze_mixed_model_anova(aclew_scores, dvs, predictors)


LENA®

=== miss ===

Full Model R² (marginal) = 9.7%, n = 581

Significant Predictors:
Predictor                          β        sr²         p-value
-----------------------------------------------------------------
voc_dur_chi                   -0.306       2.4%       p<.001***
voc_dur_och                    0.311       0.8%        p=0.014*

=== false_alarm ===


/home/engaclew/miniconda3/envs/neurogen/lib/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:508: RuntimeWarning: invalid value encountered in subtract
  return rhs / s - ql / s**2
/home/engaclew/miniconda3/envs/neurogen/lib/python3.10/site-packages/numpy/linalg/linalg.py:2120: RuntimeWarning: invalid value encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)



Full Model R² (marginal) = nan%, n = 610


/home/engaclew/miniconda3/envs/neurogen/lib/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:508: RuntimeWarning: invalid value encountered in subtract
  return rhs / s - ql / s**2
/home/engaclew/miniconda3/envs/neurogen/lib/python3.10/site-packages/numpy/linalg/linalg.py:2120: RuntimeWarning: invalid value encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)
/home/engaclew/miniconda3/envs/neurogen/lib/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:508: RuntimeWarning: invalid value encountered in subtract
  return rhs / s - ql / s**2
/home/engaclew/miniconda3/envs/neurogen/lib/python3.10/site-packages/numpy/linalg/linalg.py:2120: RuntimeWarning: invalid value encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)
/home/engaclew/miniconda3/envs/neurogen/lib/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:508: RuntimeWarning: invalid value encountered in subt


No significant predictors

=== confusion ===

Full Model R² (marginal) = 5.3%, n = 581

No significant predictors

=== correct ===

Full Model R² (marginal) = 7.9%, n = 581

Significant Predictors:
Predictor                          β        sr²         p-value
-----------------------------------------------------------------
voc_dur_chi                    0.261       1.6%       p<.001***
voc_dur_och                   -0.224      -0.3%        p=0.045*

=== CTC_mape ===

Full Model R² (marginal) = 30.8%, n = 750

Significant Predictors:
Predictor                          β        sr²         p-value
-----------------------------------------------------------------
voc_dur_chi                    1.061       7.1%       p<.001***
voc_dur_fem                    0.705       6.2%       p<.001***
voc_dur_och                    1.259       4.2%       p<.001***
voc_dur_mal                    1.191       3.2%       p<.001***

=== AWC_mape ===

Full Model R² (marginal) = 1.2%, n = 750

No signifi